In [3]:
!pip install optuna
!pip install -U kaleido
!pip install matplotlib
!pip install requests
!pip install dotenv

In [4]:
import numpy as np
import pandas as pd
import torch

In [5]:
from sklearn.preprocessing import OrdinalEncoder

class Encoder:

    def __init__(self) -> None:
        self.user_encoder = OrdinalEncoder(dtype=int, handle_unknown='use_encoded_value', unknown_value=-1)
        self.item_encoder = OrdinalEncoder(dtype=int, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, interactions: pd.DataFrame) -> pd.DataFrame:
        self.user_encoder.fit(
            interactions['user_id'].values.reshape(-1, 1)
        )
        self.item_encoder.fit(
            interactions['item_id'].values.reshape(-1, 1)
        )
        return self

    def transform(self, interactions: pd.DataFrame) -> pd.DataFrame:
        interactions = interactions.copy()
        interactions.loc[:, 'user_id'] = self.user_encoder.transform(
            interactions['user_id'].values.reshape(-1, 1)
        ).ravel()
        interactions.loc[:, 'item_id'] = self.item_encoder.transform(
            interactions['item_id'].values.reshape(-1, 1)
        ).ravel()
        if (interactions['user_id'] == -1).any() or (interactions['item_id'] == -1).any():
            print(f'Found {len(interactions[interactions["user_id"] == -1])} unknown users!')
            print(f'Found {len(interactions[interactions["item_id"] == -1])} unknown items!')
        interactions = interactions[
            (interactions['user_id'] != -1) &
            (interactions['item_id'] != -1)
        ]
        return interactions

    def fit_transform(self, interactions: pd.DataFrame) -> pd.DataFrame:
        return self.fit(interactions).transform(interactions)

    def inverse_transform(self, interactions: pd.DataFrame) -> pd.DataFrame:
        interactions.loc[:, 'user_id'] = self.user_encoder.inverse_transform(
            interactions['user_id'].values.reshape(-1, 1)
        ).ravel()
        interactions.loc[:, 'item_id'] = self.item_encoder.inverse_transform(
            interactions['item_id'].values.reshape(-1, 1)
        ).ravel()
        return interactions

In [6]:
from functools import cached_property
from scipy.sparse import csr_matrix, vstack, hstack, diags

class Dataset:
    def __init__(self, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self._check_integrity()

    def _check_integrity(self) -> None:
        ordinal_series = [
            self.train_df['user_id'],
            self.train_df['item_id'],
        ]
        for series in ordinal_series:
            assert series.min() == 0
            assert series.max() == series.nunique() - 1

    @cached_property
    def user_cnt(self) -> int:
        return self.train_df['user_id'].nunique()

    @cached_property
    def item_cnt(self) -> int:
        return self.train_df['item_id'].nunique()

    @cached_property
    def interaction_cnt(self) -> int:
        return len(self.train_df)

    @cached_property
    def density(self) -> float:
        return self.interaction_cnt / (self.user_cnt * self.item_cnt)

    @cached_property
    def user_item_matrix(self) -> csr_matrix:
        user_ids = self.train_df['user_id']
        item_ids = self.train_df['item_id']
        data = np.ones_like(user_ids)
        user_item_matrix = csr_matrix((data, (user_ids, item_ids)), shape=(self.user_cnt, self.item_cnt))
        return user_item_matrix

    @cached_property
    def adj_matrix(self) -> csr_matrix:
        return self.user_item_matrix

    @cached_property
    def extended_adj_matrix(self) -> csr_matrix:
        upper_left_zeros = csr_matrix((self.user_cnt, self.user_cnt))
        upper_part = hstack([upper_left_zeros, self.adj_matrix])
        lower_right_zeros = csr_matrix((self.item_cnt, self.item_cnt))
        lower_part = hstack([self.adj_matrix.transpose(), lower_right_zeros])
        extended_adj_matrix = vstack([upper_part, lower_part])
        return extended_adj_matrix

    @cached_property
    def normalized_matrix(self) -> csr_matrix:
        row_sum = np.array(self.extended_adj_matrix.sum(axis=1)).squeeze()
        row_sum[row_sum == 0] = 1.0
        normalizer = diags(row_sum ** -0.5)
        normalized_matrix = normalizer @ self.extended_adj_matrix @ normalizer
        return normalized_matrix


In [7]:
import os

def get_dataset(dataset_name: str) -> Dataset:

    def is_colab():
        return "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
    if is_colab():
        from google.colab import drive
        drive.mount('/content/drive')
        shns_data_path = '/content/drive/MyDrive/SHNS_data'
    else:
        shns_data_path = './SHNS_data'

    data_path = shns_data_path + '/' + dataset_name
    def load_train_df(path):
        rows = []
        with open(path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                assert len(parts) == 2
                user_id = int(parts[0])
                item_id = int(parts[1])
                rows.append({
                    'user_id': user_id,
                    'item_id': item_id,
                })
        df = pd.DataFrame(rows, columns=['user_id', 'item_id'])
        return df

    def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
        num_duplicates = df.duplicated(subset=["user_id", "item_id"]).sum()
        if num_duplicates > 0:
            print(f"Found {num_duplicates} duplicate rows")
            df = df.drop_duplicates(subset=["user_id", "item_id"]).reset_index(drop=True)
        else:
            print("No duplicate rows found")
        return df

    train_df = load_train_df(data_path + '/train.txt')
    valid_df = load_train_df(data_path + '/valid.txt')
    test_df = load_train_df(data_path + '/test.txt')
    train_df = remove_duplicates(train_df)
    valid_df = remove_duplicates(valid_df)
    test_df = remove_duplicates(test_df)
    encoder = Encoder()
    train_df = encoder.fit_transform(train_df)
    valid_df = encoder.transform(valid_df)
    test_df = encoder.transform(test_df)
    return Dataset(train_df, valid_df, test_df)

In [8]:
class LightGCN(torch.nn.Module):
    def __init__(self, dataset: Dataset, hyperparams: dict) -> None:
        super(LightGCN, self).__init__()
        self.dataset = dataset
        self.hyperparams = hyperparams
        self.user_embeddings = torch.nn.Embedding(
            self.dataset.user_cnt, self.hyperparams['emb_size']
        )
        self.item_embeddings = torch.nn.Embedding(
            self.dataset.item_cnt, self.hyperparams['emb_size']
        )
        self.train_trues = self.dataset.train_df.groupby('user_id')['item_id'].apply(np.array)
    
        # for ANS (+ DENS)
        self.user_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])
        self.item_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])
        self.margin_model = torch.nn.Linear(self.hyperparams['emb_size'], 1)
        self.pos_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])
        self.neg_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])

        torch.nn.init.xavier_uniform_(self.user_embeddings.weight)
        torch.nn.init.xavier_uniform_(self.item_embeddings.weight)

        self.aggregator = self.get_aggregator()

    def forward(self, user_indices: torch.Tensor, item_indices: torch.Tensor) -> torch.Tensor:
        user_embeddings, item_embeddings = self.get_embeddings()
        user_embeddings = user_embeddings[user_indices]
        item_embeddings = item_embeddings[item_indices]
        return torch.sum(user_embeddings * item_embeddings, dim=1)

    def get_embeddings(self, aggregate=True) -> tuple[torch.Tensor, torch.Tensor]:
        embeddings = []
        full_embedding = torch.cat([self.user_embeddings.weight, self.item_embeddings.weight], dim=0)
        embeddings.append(full_embedding)
        for _ in range(self.hyperparams['layer_num']):
            full_embedding = torch.sparse.mm(self.aggregator, full_embedding)
            embeddings.append(full_embedding)
        final_embedding = torch.stack(embeddings, dim=0)
        if aggregate:
            final_embedding = torch.mean(final_embedding, dim=0)
        else:
            final_embedding = torch.permute(final_embedding, (1, 0, 2))
        final_user_embedding, final_item_embedding = torch.split(
            final_embedding, [self.dataset.user_cnt, self.dataset.item_cnt],
        )
        return final_user_embedding, final_item_embedding

    def get_aggregator(self) -> torch.Tensor:
        coo = self.dataset.normalized_matrix.tocoo()
        indices = torch.tensor(np.array([coo.row, coo.col]), dtype=torch.long)
        values = torch.tensor(coo.data, dtype=torch.float32)
        aggregator = torch.sparse_coo_tensor(indices, values, size=coo.shape)
        return aggregator

    
    def get_topk(self, k: int) -> torch.Tensor:
        user_embeddings, item_embeddings = self.get_embeddings()
        device = user_embeddings.device
        n_users = user_embeddings.size(0)
        batch_size = 10000

        item_emb_T = item_embeddings.T
        csr = self.dataset.user_item_matrix  # scipy.sparse.csr_matrix

        topk_list = []
        neg_inf = torch.tensor(float("-inf"), device=device)

        for start in range(0, n_users, batch_size):
            end = min(start + batch_size, n_users)
            B = end - start

            batch_user_emb = user_embeddings[start:end]
            scores = batch_user_emb @ item_emb_T  # [B, n_items]

            sub = csr[start:end]          # CSR slice
            indptr = sub.indptr           # shape [B+1]
            cols = sub.indices            # all nnz cols in this block

            # row id를 nnz 개수만큼 반복해서 (row, col) 좌표 만들기
            row = np.repeat(np.arange(B), np.diff(indptr))

            # torch index로 옮겨서 해당 위치만 -inf 처리
            row_t = torch.from_numpy(row).to(device=device, dtype=torch.long, non_blocking=True)
            col_t = torch.from_numpy(cols).to(device=device, dtype=torch.long, non_blocking=True)
            scores[row_t, col_t] = neg_inf

            topk_list.append(torch.topk(scores, k=k, dim=1).indices)

        return torch.cat(topk_list, dim=0)


    def to(self, device: torch.device):
        super(LightGCN, self).to(device)
        self.aggregator = self.aggregator.to(device)
        return self

In [ ]:
import numpy as np
import torch
from scipy.sparse import csr_matrix

class Sampler:
    def __init__(self, model, dataset, hyperparams: dict) -> None:
        self.model = model
        self.dataset = dataset
        self.hyperparams = hyperparams
        self.device = "cuda"

        self.U = int(dataset.user_cnt)
        self.I = int(dataset.item_cnt)
        self.C = int(hyperparams["cand_size"])
        self.max_round = int(hyperparams.get("sampler_max_round", 50))

        adj: csr_matrix = dataset.user_item_matrix.tocsr()
        indptr = adj.indptr.astype(np.int64)
        indices = adj.indices.astype(np.int64)

        users_edge = np.repeat(np.arange(self.U, dtype=np.int64), np.diff(indptr))
        pos_edge = indices.astype(np.int64)

        self.users_edge = torch.from_numpy(users_edge).long()
        self.pos_edge = torch.from_numpy(pos_edge).long()
        self.K = int(self.pos_edge.numel())

        key_pos = users_edge * self.I + pos_edge
        self.key_pos_sorted = torch.from_numpy(np.sort(key_pos)).to(self.device)

    @torch.no_grad()
    def _exists_in_pos(self, keys_query: torch.Tensor) -> torch.Tensor:
        flat = keys_query.reshape(-1)
        idx = torch.searchsorted(self.key_pos_sorted, flat)
        idx = idx.clamp(0, self.key_pos_sorted.numel() - 1)
        hit = self.key_pos_sorted[idx] == flat
        return hit.view_as(keys_query)

    @torch.no_grad()
    def get_samples(self, batch_idx: torch.Tensor) -> torch.Tensor:
        if batch_idx.device.type != "cpu":
            batch_idx = batch_idx.detach().cpu()

        users = self.users_edge[batch_idx].to(self.device, non_blocking=True)
        pos = self.pos_edge[batch_idx].to(self.device, non_blocking=True)

        B = int(users.numel())
        neg = torch.randint(0, self.I, (B, self.C), device=self.device, dtype=torch.long)

        def conflict_mask(neg_):
            same_pos = neg_.eq(pos.unsqueeze(1))
            key = users.unsqueeze(1).to(torch.int64) * self.I + neg_.to(torch.int64)
            in_pos = self._exists_in_pos(key)
            return same_pos | in_pos

        conflict = conflict_mask(neg)
        r = 0
        while conflict.any():
            r += 1
            if r > self.max_round:
                break
            m = conflict
            neg[m] = torch.randint(0, self.I, (int(m.sum().item()),), device=self.device)
            conflict = conflict_mask(neg)

        return torch.cat([users.view(B, 1), pos.view(B, 1), neg], dim=1)


In [10]:
import math

def get_recall(pred: list[list], true: list[list]) -> float:
    total_recall = 0.0
    valid_cnt = 0
    for p, t in zip(pred, true):
        if len(t) == 0:
            continue
        hit = len(set(p) & set(t))
        total_recall += hit / len(t)
        valid_cnt += 1
    return total_recall / valid_cnt if valid_cnt > 0 else -1

def get_ndcg(pred: list[list], true: list[list]) -> float:
    total_ndcg = 0.0
    valid_cnt = 0
    for p, t in zip(pred, true):
        if len(t) == 0:
            continue
        dcg = 0.0
        for idx, item in enumerate(p):
            if item in t:
                dcg += 1 / math.log2(idx + 2)
        ideal_len = min(len(t), len(p))
        idcg = sum(1 / math.log2(i + 2) for i in range(ideal_len))
        total_ndcg += dcg / idcg if idcg > 0 else 0.0
        valid_cnt += 1
    return total_ndcg / valid_cnt if valid_cnt > 0 else -1

In [11]:
import matplotlib.pyplot as plt
import math

def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.log(1 + torch.exp(neg_scores - pos_scores)))

def model_reg_loss(user_embs: torch.Tensor, pos_embs: torch.Tensor, neg_embs: torch.Tensor) -> torch.Tensor:
    reg_loss = torch.norm(user_embs) ** 2 + torch.norm(pos_embs) ** 2 + torch.norm(neg_embs) ** 2
    reg_loss /= len(user_embs)
    return reg_loss

class Trainer:
    def __init__(self, dataset: Dataset):
        self.dataset = dataset
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # self.device = torch.device('cpu')
        valid_trues = self.dataset.val_df.groupby('user_id')['item_id'].apply(list).to_dict()
        self.valid_trues = [valid_trues[i] if i in valid_trues else [] for i in range(self.dataset.user_cnt)]
        test_trues = self.dataset.test_df.groupby('user_id')['item_id'].apply(list).to_dict()
        self.test_trues = [test_trues[i] if i in test_trues else [] for i in range(self.dataset.user_cnt)]

    def dns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)
        neg_indices = torch.argmax(user_neg, dim=1).detach()
        neg = neg_cand[torch.arange(neg_cand.size(0)), neg_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def dns_mn_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        B, C = neg_cand.size()
        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)

        m = int(self.hyperparams['m_ratio'] * (C - 1)) + 1
        if m == 1:
            neg_rank = torch.zeros((B,), device=self.device, dtype=torch.long)
        else:
            neg_rank = torch.randint(0, m, (B,), device=self.device)
        neg_cand_indices = torch.argsort(user_neg, dim=1, descending=True)[torch.arange(B), neg_rank]
        neg = neg_cand[torch.arange(neg_cand.size(0)), neg_cand_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def shns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:

        B, C = neg_cand.size()
        def stochastic_round(x: torch.Tensor) -> torch.Tensor:
            lower = torch.floor(x)
            prob_up = x - lower
            rand = torch.rand_like(x)
            rounded = torch.where(rand < prob_up, lower + 1, lower)
            return rounded.to(torch.long)

        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)

        if 'alpha' in self.hyperparams:
            alpha = self.hyperparams['alpha']
        elif 'alpha_step' in self.hyperparams:
            alpha = self.hyperparams['alpha_step'] * self.hyperparams['smooth_epoch']
        neg_rank = alpha * (C - 1)
        neg_rank = torch.full((B,), neg_rank, dtype=torch.float32)
        neg_rank = torch.clamp(neg_rank, min=0, max=(C - 1))
        neg_rank = stochastic_round(neg_rank)

        neg_cand_indices = torch.argsort(user_neg, dim=1, descending=True)[torch.arange(B), neg_rank]
        neg = neg_cand[torch.arange(B), neg_cand_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]

        pos_weight = torch.exp(user_embeddings[users] * pos_embeddings)
        neg_weight = torch.exp(user_embeddings[users] * neg_embeddings)
        interp_weight = neg_weight / (neg_weight + self.hyperparams['beta'] * pos_weight)
        mixed_neg_embeddings = neg_embeddings * interp_weight + pos_embeddings * (1 - interp_weight)

        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * mixed_neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def ahns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        user_embeddings, item_embeddings = model.get_embeddings()
        target_diffs = self.hyperparams['beta'] * (
            (user_embeddings[users] * item_embeddings[pos]).sum(dim=1) + self.hyperparams['alpha']
        ) ** -1

        diffs = (user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]).sum(dim=-1)
        ratings = torch.abs(target_diffs.unsqueeze(-1) - diffs)
        neg_indices = torch.argmin(ratings, dim=1)
        neg = neg_cand[torch.arange(neg_cand.size(0)), neg_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def pure_shns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        B, C = neg_cand.size()
        def stochastic_round(x: torch.Tensor) -> torch.Tensor:
            lower = torch.floor(x)
            prob_up = x - lower
            rand = torch.rand_like(x)
            rounded = torch.where(rand < prob_up, lower + 1, lower)
            return rounded.to(torch.long)

        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)

        if 'alpha' in self.hyperparams:
            alpha = self.hyperparams['alpha']
        elif 'alpha_step' in self.hyperparams:
            alpha = self.hyperparams['alpha_step'] * self.hyperparams['smooth_epoch']
        neg_rank = alpha * (C - 1)
        neg_rank = torch.full((B,), neg_rank, dtype=torch.float32)
        neg_rank = torch.clamp(neg_rank, min=0, max=(C - 1))
        neg_rank = stochastic_round(neg_rank)

        neg_cand_indices = torch.argsort(user_neg, dim=1, descending=True)[torch.arange(B), neg_rank]
        neg = neg_cand[torch.arange(B), neg_cand_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def dins_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        def pooling(embeddings):
            return embeddings.mean(dim=1)

        user, pos_item, neg_candidates = users, pos, neg_cand
        user_gcn_emb, item_gcn_emb = model.get_embeddings(aggregate=False)

        # code from https://github.com/Wu-Xi/DINS/blob/main/modules/LightGCN_DINS.py

        batch_size = user.shape[0]
        s_e, p_e = user_gcn_emb[user], item_gcn_emb[pos_item]  # [batch_size, n_hops+1, channel]
        s_e = pooling(s_e).unsqueeze(dim=1)

        """Hard Boundary Definition"""
        n_e = item_gcn_emb[neg_candidates]  # [batch_size, n_negs, n_hops, channel]
        scores = (s_e.unsqueeze(dim=1) * n_e).sum(dim=-1)  # [batch_size, n_negs, n_hops+1]
        indices = torch.max(scores, dim=1)[1].detach()  # torch.Size([2048, 3])
        neg_items_emb_ = n_e.permute([0, 2, 1, 3])  # [batch_size, n_hops+1, n_negs, channel]
        neg_items_embedding_hardest = neg_items_emb_[[[i] for i in range(batch_size)],range(neg_items_emb_.shape[1]), indices, :]   #   [batch_size, n_hops+1, channel]

        """Dimension Independent Mixup"""
        neg_scores = torch.exp(s_e *neg_items_embedding_hardest)  # [batch_size, n_hops, channel]
        total_sum = self.hyperparams['beta'] * torch.exp ((s_e * p_e))+neg_scores   # [batch_size, n_hops, channel]
        neg_weight = neg_scores/total_sum     # [batch_size, n_hops, channel]
        pos_weight = 1-neg_weight   # [batch_size, n_hops, channel]

        n_e_ =  pos_weight * p_e + neg_weight * neg_items_embedding_hardest  # mixing

        neg_gcn_embs = n_e_
        neg_gcn_embs = neg_gcn_embs.unsqueeze(dim=1)
        user_gcn_emb = user_gcn_emb[user]
        pos_gcn_embs = item_gcn_emb[pos_item]

        u_e = pooling(user_gcn_emb)
        pos_e = pooling(pos_gcn_embs)
        neg_e = pooling(neg_gcn_embs.view(-1, neg_gcn_embs.shape[2], neg_gcn_embs.shape[3])).view(batch_size, 1, -1)

        pos_scores = torch.sum(torch.mul(u_e, pos_e), axis=1)
        neg_scores = torch.sum(torch.mul(u_e.unsqueeze(dim=1), neg_e), axis=-1)  # [batch_size, K]
        loss = bpr_loss(pos_scores, neg_scores.squeeze())
        reg_loss = model_reg_loss(user_gcn_emb[:, 0, :], pos_gcn_embs[:,0, :], neg_gcn_embs[:, :, 0, :])
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def ans_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        user, pos_item, neg_item = users, pos, neg_cand
        user_all_embeddings, item_all_embeddings = model.get_embeddings(aggregate=False)

        # code from https://github.com/Asa9aoTK/ANS-Recbole/blob/main/recbole/model/general_recommender/ans.py
        u_embeddings = user_all_embeddings[user]
        pos_embeddings = item_all_embeddings[pos_item]
        neg_embeddings = item_all_embeddings[neg_item]

        s_e = u_embeddings
        p_e = pos_embeddings
        n_e = neg_embeddings
        batch_size = user.shape[0]

        gate_neg_hard = torch.sigmoid(model.item_gate(n_e) * model.user_gate(s_e).unsqueeze(1))
        n_hard =  n_e * gate_neg_hard
        n_easy =  n_e - n_hard

        p_hard =  p_e.unsqueeze(1) * gate_neg_hard
        p_easy =  p_e.unsqueeze(1) - p_hard

        import torch.nn.functional as F
        distance = torch.mean(F.pairwise_distance(n_hard, p_hard, p=2).squeeze(dim=1))
        temp = torch.norm(torch.mul(p_easy, n_easy),dim=-1)
        orth = torch.mean(torch.sum(temp,axis=-1))

        margin = torch.sigmoid(1/model.margin_model(n_hard * p_hard))

        random_noise = torch.rand(n_easy.shape).to(self.device)
        magnitude = torch.nn.functional.normalize(random_noise, p=2, dim=-1) * margin *0.1
        direction = torch.sign(p_easy - n_easy)
        noise = torch.mul(direction,magnitude)
        n_easy_syth = noise + n_easy
        n_e_ = n_hard + n_easy_syth
        hard_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_hard), axis=-1)  # [batch_size, K]
        easy_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_easy), axis=-1)  # [batch_size, K]
        syth_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_e_), axis=-1)  # [batch_size, K]
        norm_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_e), axis=-1)  # [batch_size, K]
        sns_loss = torch.mean(torch.log(1 + torch.exp(easy_scores - hard_scores).sum(dim=1)))
        dis_loss = distance + orth
        scores = (s_e.unsqueeze(dim=1) * n_e_).sum(dim=-1)  # [batch_size, n_negs]
        scores_false =  syth_scores - norm_scores

        indices = torch.max(scores + self.hyperparams['epsilon']*scores_false, dim=1)[1].detach()
        neg_items_emb_ = n_e_.permute([0, 2, 1, 3])  # [batch_size, n_hops+1, n_negs, channel]
        # [batch_size, n_hops+1, channel]
        neg_embeddings = neg_items_emb_[[[i] for i in range(batch_size)],range(neg_items_emb_.shape[1]), indices, :]

        # calculate BPR Loss
        pos_scores = torch.mul(u_embeddings, pos_embeddings).sum(dim=1).squeeze(dim=1).sum(dim=-1)
        neg_scores = torch.mul(u_embeddings, neg_embeddings).sum(dim=1).sum(dim=1)
        mf_loss = bpr_loss(pos_scores, neg_scores)

        # calculate BPR Loss
        u_ego_embeddings = model.user_embeddings(user)
        pos_ego_embeddings = model.item_embeddings(pos_item)
        neg_ego_embeddings = model.item_embeddings(neg_item)

        loss = mf_loss + self.hyperparams['gamma'] * (sns_loss + dis_loss)
        reg_loss = model_reg_loss(u_ego_embeddings, pos_ego_embeddings, neg_ego_embeddings)
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def train_model(self, model: LightGCN, sampler: Sampler, hyperparams: dict) -> LightGCN:
        self.hyperparams = hyperparams
        model = model.to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        idx_loader = torch.utils.data.DataLoader(
            torch.arange(sampler.K),
            batch_size=hyperparams["batch_size"],
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=4
        )

        for (step, batch_idx) in enumerate(idx_loader):
            cur_batch = sampler.get_samples(batch_idx)
            self.hyperparams['smooth_epoch'] = self.hyperparams['epoch'] + (step / len(idx_loader))
            optimizer.zero_grad()
            users = cur_batch[:, 0]
            pos = cur_batch[:, 1]
            neg_cand = cur_batch[:, 2:]
            loss = getattr(self, self.hyperparams['method'] + '_loss')(model, users, pos, neg_cand)
            loss.backward()
            optimizer.step()
        return model

    def validate(self, model: LightGCN) -> tuple:
        result = {}
        for topk in [20]:
            model.eval()
            with torch.no_grad():
                preds = model.get_topk(topk).to('cpu').numpy().tolist()
            model.train()

            cur_recall = get_recall(preds, self.valid_trues)
            cur_ndcg = get_ndcg(preds, self.valid_trues)
            result[f'recall_{topk}'] = cur_recall
            result[f'ndcg_{topk}'] = cur_ndcg
        return result

    def test(self, model: LightGCN) -> tuple:
        result = {}
        for topk in [20]:
            model.eval()
            with torch.no_grad():
                preds = model.get_topk(topk).to('cpu').numpy().tolist()
            model.train()

            cur_recall = get_recall(preds, self.test_trues)
            cur_ndcg = get_ndcg(preds, self.test_trues)
            result[f'recall_{topk}'] = cur_recall
            result[f'ndcg_{topk}'] = cur_ndcg
        return result

In [12]:
import torch

def train(dataset: Dataset, hyperparams: dict, print_result: bool = False):
    trainer = Trainer(dataset)
    test_model = LightGCN(dataset, hyperparams).to('cuda')

    best_val_result = None
    best_test_result = None
    best_recall = -1e10
    patience = 10

    sampler = Sampler(test_model, dataset, hyperparams)
    for epoch in range(100):
        hyperparams['epoch'] = epoch
        trainer.train_model(test_model, sampler, hyperparams)
        val_result = trainer.validate(test_model)
        test_result = trainer.test(test_model)

        cur_recall = val_result['recall_20']
        if print_result:
            print(cur_recall)
        if cur_recall > best_recall:
            best_recall = cur_recall
            best_val_result = val_result
            best_test_result = test_result
            patience = 10
        else:
            patience -= 1
            if patience == 0:
                break
    return test_model, best_val_result, best_test_result

In [13]:
import optuna
import gc

def search_hyperparams(dataset_name: str, method: str, base_hyperparams: dict, n_trials: int = 50) -> optuna.Study:

    def get_hyperparams(trial: optuna.Trial) -> dict:
        hyperparams = base_hyperparams.copy()
        hyperparams['method'] = method
        hyperparams['cand_size'] = 2 ** trial.suggest_int('cand_size_exp', low=1, high=9, step=1)

        if hyperparams['method'] == 'ans':
            hyperparams['gamma'] = trial.suggest_float('gamma', low=0.01, high=0.50, step=0.01)
            hyperparams['epsilon'] = trial.suggest_float('epsilon', low=0.01, high=1.00, step=0.01)
        elif hyperparams['method'] == 'shns':
            hyperparams['alpha_step'] = 1e-5 * 2 ** trial.suggest_int('alpha_step_exp', low=1, high=9, step=1)
            hyperparams['beta'] = trial.suggest_float('beta', low=0.1, high=10.0, step=0.1)
        elif hyperparams['method'] == 'pure_shns':
            hyperparams['alpha_step'] = 1e-5 * 2 ** trial.suggest_int('alpha_step_exp', low=1, high=9, step=1)
        elif hyperparams['method'] == 'ahns':
            hyperparams['alpha'] = trial.suggest_float('alpha', low=0.1, high=1.0, step=0.1)
            hyperparams['beta'] = trial.suggest_float('beta', low=0.1, high=1.0, step=0.1)
        elif hyperparams['method'] == 'dins':
            hyperparams['beta'] = trial.suggest_float('beta', low=0.1, high=10.0, step=0.1)
        elif hyperparams['method'] == 'dns_mn':
            hyperparams['m_ratio'] = trial.suggest_float('m_ratio', low=0.01, high=0.50, step=0.01)
        elif hyperparams['method'] == 'dns':
            pass
        else:
            raise ValueError(f'Unknown method: {hyperparams["method"]}')
        return hyperparams

    def objective(trial, dataset: Dataset):
        gc.collect()
        torch.cuda.empty_cache()
        hyperparams = get_hyperparams(trial)
        print(hyperparams)
        _, best_val_result, best_test_result = train(dataset, hyperparams, print_result=True)
        print(f'test: {best_test_result}')
        return best_val_result['recall_20']

    dataset = get_dataset(dataset_name)
    study_name = f'{dataset_name}_dataset_{method}_layer_num_{base_hyperparams["layer_num"]}'
    sampler = optuna.samplers.TPESampler()
    study = optuna.create_study(
        study_name=study_name,
        direction='maximize',
        sampler=sampler
    )
    study.optimize(lambda trial: objective(trial, dataset), n_trials=n_trials)
    return study

In [14]:
import copy
import math

def get_test_result(dataset_name: str, hyperparams: dict, n_trials: int = 10) -> dict:
    def is_number(x):
        return isinstance(x, (int, float)) and not (isinstance(x, float) and math.isnan(x))

    avg_best_test_result = None
    prev_good_result = None
    valid_trials = 0
    dataset = get_dataset(dataset_name)

    for _ in range(n_trials):
        _, _, best_test_result = train(dataset, hyperparams, print_result=True)
        print(best_test_result)

        cur = best_test_result
        cur_recall = cur["recall_20"]
        prev_recall = None if prev_good_result is None else prev_good_result["recall_20"]

        is_outlier = (
            prev_good_result is not None
            and is_number(cur_recall)
            and is_number(prev_recall)
            and prev_recall > 0
            and cur_recall < 0.5 * prev_recall
        )

        if is_outlier:
            continue

        prev_good_result = cur
        if avg_best_test_result is None:
            avg_best_test_result = {k: float(v) for k, v in cur.items() if is_number(v)}
        else:
            for k, v in cur.items():
                if is_number(v):
                    avg_best_test_result[k] = avg_best_test_result.get(k, 0.0) + float(v)

        valid_trials += 1

    for k in list(avg_best_test_result.keys()):
        avg_best_test_result[k] /= valid_trials

    return avg_best_test_result

In [15]:
import os
import tempfile
import requests
from dotenv import load_dotenv
load_dotenv()

import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import plot_slice


NOTION_VERSION = "2025-09-03"

def save_to_notion(study, test_result: dict):
    token = os.getenv("NOTION_API_TOKEN")
    page_id = os.getenv("NOTION_PAGE_ID")

    ax = plot_slice(study)
    fig = ax[0].figure

    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    tmp_path = tmp.name
    tmp.close()

    fig.savefig(tmp_path, dpi=200)
    plt.close(fig)

    h_json = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
        "Content-Type": "application/json",
    }
    h_send = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
    }

    r = requests.post(
        "https://api.notion.com/v1/file_uploads",
        headers=h_json,
        json={"filename": "plot_slice.png", "content_type": "image/png"},
    )
    r.raise_for_status()
    upload_id = r.json()["id"]

    with open(tmp_path, "rb") as f:
        r = requests.post(
            f"https://api.notion.com/v1/file_uploads/{upload_id}/send",
            headers=h_send,
            files={"file": ("plot_slice.png", f, "image/png")},
        )
    r.raise_for_status()

    test_str = "\n".join(f"{k}: {v}" for k, v in test_result.items())
    best_str = "\n".join(f"{k}: {v}" for k, v in study.best_params.items())

    payload = {
        "children": [
            {"object": "block", "type": "divider", "divider": {}},
            {
                "object": "block",
                "type": "heading_3",
                "heading_3": {
                    "rich_text": [{"type": "text", "text": {"content": f"{study.study_name} (best={study.best_value})"}}]
                },
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "best_params\n" + best_str}}]},
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "test_result\n" + test_str}}]},
            },
            {
                "object": "block",
                "type": "image",
                "image": {"type": "file_upload", "file_upload": {"id": upload_id}},
            },
        ]
    }

    r = requests.patch(
        f"https://api.notion.com/v1/blocks/{page_id}/children",
        headers=h_json,
        json=payload,
    )
    r.raise_for_status()


In [ ]:
methods = ['dins', 'pure_shns', 'shns']
layer_nums = [0, 2]
dataset_names = ['tool', 'home', 'baby', 'ali']
for layer_num in layer_nums:
    for dataset_name in dataset_names:
        for method in methods:
            print(f'dataset: {dataset_name}, method: {method}, layer_num: {layer_num}')
            base_hyperparams= {
                'layer_num': layer_num,
                'reg': 1e-5,
                'batch_size': 512,
                'emb_size': 32,
            }
            study = search_hyperparams(dataset_name, method, base_hyperparams, n_trials=50)
            print(f'best params: {study.best_params}')
            print(f'best value: {study.best_value}')
            best_hyperparams = base_hyperparams.copy()
            best_hyperparams.update(study.best_params)
            best_hyperparams['method'] = method
            best_hyperparams['cand_size'] = 2 ** best_hyperparams['cand_size_exp']
            if 'alpha_step_exp' in best_hyperparams:
                best_hyperparams['alpha_step'] = 1e-5 * 2 ** best_hyperparams['alpha_step_exp']

            test_result = get_test_result(dataset_name, best_hyperparams, n_trials=10)
            print(f'test result: {test_result}')
            save_to_notion(study, test_result)

dataset: ml1m, method: pure_shns, layer_num: 0


[I 2026-02-05 15:42:49,731] A new study created in memory with name: ml1m_dataset_pure_shns_layer_num_0


No duplicate rows found
No duplicate rows found
No duplicate rows found
Found 0 unknown users!
Found 17 unknown items!
Found 0 unknown users!
Found 20 unknown items!
{'layer_num': 0, 'reg': 1e-05, 'batch_size': 512, 'emb_size': 32, 'method': 'pure_shns', 'cand_size': 4, 'alpha_step': 0.00016}
0.13547310742365207
0.16178287497719682
0.1853709821236286
0.20009270143577298
0.2130689319071755
0.21935753494725452
0.22362863115915535
0.22967657688104914
0.2309960055206164
0.23370117127872275
0.23518825728625112
0.23746274940441195
0.2378264536973504
0.2381077049903675
0.23843406324924782
0.24000120956474144
0.2394596792275473
0.24009534493695212
0.24067274880666445
0.2388082767716625
0.24095158515583873
0.24077959649204925
0.23984935360407847
0.2407737152243185
0.2381948660563939
0.23763781631149666
0.23739836033085482
0.23708632558659892
0.2381607737159204
0.23685865401869458


[I 2026-02-05 15:43:47,478] Trial 0 finished with value: 0.24095158515583873 and parameters: {'cand_size_exp': 2, 'alpha_step_exp': 4}. Best is trial 0 with value: 0.24095158515583873.


0.2360233547274542
test: {'recall_20': 0.24137539972752184, 'ndcg_20': 0.2302222515122302}
{'layer_num': 0, 'reg': 1e-05, 'batch_size': 512, 'emb_size': 32, 'method': 'pure_shns', 'cand_size': 8, 'alpha_step': 0.00016}
0.14457935925006288
0.17313861461498053
0.1980950378048326
0.21350861429289858
0.2229206428761434
0.23133900445011726
0.2394278278304202
0.24468169625434733
0.2472048347224567
0.24910679319211018
0.25234925054165264
0.25444793467603216
0.2541004936571951
0.2551944421901514
0.25391844977500605
0.2540378260950745
0.2549147666006209
0.2556966013021412
0.25460061013471963
0.2540039112922698
0.2529706316642552
0.25242737295712514
0.25233657189720116
0.250763149760198
0.25084253537134443
0.24956411214405466
0.24988287989412922


...

0.16696254758392248
test: {'recall_20': 0.16639754484372082, 'ndcg_20': 0.1463424704784877}
{'layer_num': 0, 'reg': 1e-05, 'batch_size': 512, 'emb_size': 32, 'method': 'shns', 'cand_size': 2, 'alpha_step': 0.00016, 'beta': 0.2}
0.11702435028539239
0.1299074195158517
0.14475751790330343
0.156231667840776
0.16508025986796307
0.17374996276770935
0.18141605262257726
0.18839974436809848
0.1910044510823958
0.19622423674751469
0.20055925503709404
0.2046322227853426
0.2054452877059395
0.20829775969480782
0.21019492378223176
0.21122878196270709
0.21362348260145245
0.216195426606708
0.2159287763526778
0.21795115935258766
0.21979576303339654
0.21959709320761306
0.21940694546789521
0.2213971342278681
0.22117670071604442
0.22118078438818825
0.2233652395953912
0.22182979751679124
0.2241355869135235
0.22575629485408122
0.2268670130376874
0.22695792114966193
0.2264937145393125
0.22781362891303142
0.22639249557402363
0.2263136820014226
0.22682213948154384
0.2257637629807162
0.22703057236846158
0.228728

[I 2026-02-05 17:13:20,153] Trial 1 finished with value: 0.23369883147168985 and parameters: {'cand_size_exp': 1, 'alpha_step_exp': 4, 'beta': 0.2}. Best is trial 1 with value: 0.23369883147168985.


0.23238375908824543
test: {'recall_20': 0.23182806485936033, 'ndcg_20': 0.22496675876013755}
{'layer_num': 0, 'reg': 1e-05, 'batch_size': 512, 'emb_size': 32, 'method': 'shns', 'cand_size': 32, 'alpha_step': 2e-05, 'beta': 7.6}
0.07615502110279487
0.09165255820256586
0.10542735234658292
0.1195436595406939
0.12965674385180032
0.13981121661259224
0.14544384082653494
0.1579739233720762
0.15968187612725698
0.16526592151331085
0.16927886105917153
0.17095404700971764
0.1771746674577662
0.17619154283679553
0.17840536790475123
0.17814807425742374
0.18238999521522692
0.1817933696624584
0.18571441254303037
0.18431661125041637
0.1860265866225607
0.18645082353183756
0.18839227301438244
0.18764372450952166
0.19020154662977568
0.18804901395211884
0.18946778602605277
0.19016058465352353
0.19074419249599647
0.1904004885108863
0.19222564731122152
0.1920450350578421
0.19189304709631838
0.1938637991489299
0.19382342582513154
0.19357149962280276
0.19428645225183513
0.1985054699052448
0.19841619966794727
0

[I 2026-02-05 17:16:53,910] Trial 2 finished with value: 0.20797239773615622 and parameters: {'cand_size_exp': 5, 'alpha_step_exp': 1, 'beta': 7.6}. Best is trial 1 with value: 0.23369883147168985.


0.2032837295196877
test: {'recall_20': 0.21021192076296785, 'ndcg_20': 0.1820631846133059}
{'layer_num': 0, 'reg': 1e-05, 'batch_size': 512, 'emb_size': 32, 'method': 'shns', 'cand_size': 2, 'alpha_step': 0.00512, 'beta': 3.0000000000000004}
0.11741717336441969
0.11867765628595466
0.11858881232376212
0.12113325999677484
0.12203485097650477
0.12295036378868528
0.12442440749001406
0.1261969383836274
0.12812877870998757
0.1289434670360499
0.13001506685620212
0.12837730800074834
0.13002270624172696
0.12987374222934903
0.13055461843532165
0.12988191990167794
0.1313335114677399
0.12894372209830784
0.1314500482429784
